# ReviewLens: Exploratory Data Analysis (EDA)

This notebook explores the cleaned mobile game review dataset and DuckDB warehouse, evaluating data quality, rating distributions, temporal coverage, version distributions, and text length characteristics.

In [ ]:
import duckdb
import pandas as pd
import json
from pathlib import Path

# Check paths
parquet_path = Path("../data/processed/reviews.parquet")
meta_path = Path("../data/processed/meta.json")
db_path = Path("../data/warehouse/reviewlens.duckdb")

if not parquet_path.exists():
    # Fallback to local execution directory if running from repo root
    parquet_path = Path("data/processed/reviews.parquet")
    meta_path = Path("data/processed/meta.json")
    db_path = Path("data/warehouse/reviewlens.duckdb")

print(f"Parquet exists: {parquet_path.exists()}")
print(f"Meta exists: {meta_path.exists()}")
print(f"DuckDB exists: {db_path.exists()}")

## 1. Load Metadata & Summary Stats

In [ ]:
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

## 2. Load DuckDB Warehouse Data

In [ ]:
con = duckdb.connect(str(db_path), read_only=True)
df = con.execute("SELECT * FROM reviews").df()
print(f"Loaded {len(df)} rows from DuckDB warehouse.")
df.head()

## 3. Review Volume & Section 6.1 Validation

In [ ]:
summary_df = con.execute("""
    SELECT 
        game,
        COUNT(*) AS total_reviews,
        MIN(review_date) AS min_date,
        MAX(review_date) AS max_date,
        (MAX(review_date) - MIN(review_date)) AS span_days,
        ROUND(AVG(rating), 2) AS avg_rating,
        ROUND(100.0 * COUNT(app_version) / COUNT(*), 1) AS version_pct,
        ROUND(100.0 * SUM(CASE WHEN dev_replied THEN 1 ELSE 0 END) / COUNT(*), 1) AS dev_reply_pct
    FROM reviews
    GROUP BY game
""").df()
summary_df

## 4. Rating Distribution per Game

In [ ]:
rating_df = con.execute("""
    SELECT 
        game,
        rating,
        COUNT(*) AS count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY game), 1) AS pct
    FROM reviews
    GROUP BY game, rating
    ORDER BY game, rating
""").df()
rating_df

## 5. Temporal Breakdown (Weekly Volume & Rating Trends)

In [ ]:
weekly_df = con.execute("""
    SELECT 
        game,
        DATE_TRUNC('week', review_date) AS review_week,
        COUNT(*) AS weekly_count,
        ROUND(AVG(rating), 2) AS weekly_avg_rating
    FROM reviews
    GROUP BY game, review_week
    ORDER BY game, review_week
""").df()
weekly_df.head(15)

## 6. Word Count & Retrieval Indexing Eligibility (Section 6.6)

In [ ]:
word_df = con.execute("""
    SELECT 
        game,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN content_words >= 8 THEN 1 ELSE 0 END) AS indexable_rows_gte_8_words,
        ROUND(100.0 * SUM(CASE WHEN content_words >= 8 THEN 1 ELSE 0 END) / COUNT(*), 1) AS indexable_pct,
        ROUND(AVG(content_words), 1) AS avg_words,
        MEDIAN(content_words) AS median_words,
        MAX(content_words) AS max_words
    FROM reviews
    GROUP BY game
""").df()
word_df

## 7. App Version Analysis

In [ ]:
version_df = con.execute("""
    SELECT 
        game,
        app_version,
        COUNT(*) AS count,
        MIN(review_date) AS first_seen,
        MAX(review_date) AS last_seen,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM reviews
    WHERE app_version IS NOT NULL
    GROUP BY game, app_version
    ORDER BY game, count DESC
""").df()
version_df.head(20)

In [ ]:
con.close()